## Sources, Sinks & Watermarking

Notebook 15 introduced the unbounded table model and micro-batch execution. This notebook goes deeper on the three pillars that connect a streaming pipeline to the real world:

- **Sources** — where data enters Spark: files, Kafka, Rate, and socket
- **Sinks** — where results go: memory, console, files, Delta Lake, Kafka, and custom logic
- **Watermarking** — the mechanism that bounds state memory, enables late-data handling, and unlocks `append` mode for windowed aggregations

```text
┌──────────────┐    readStream     ┌───────────────────┐    writeStream    ┌──────────────┐
│    SOURCE    │ ────────────────► │  Streaming Query  │ ────────────────► │     SINK     │
│  (Kafka,     │                   │  (filter, window, │                   │  (Delta,     │
│   files,     │                   │   join, agg)      │                   │   Kafka,     │
│   rate ...)  │                   │                   │                   │   files ...) │
└──────────────┘                   └───────────────────┘                   └──────────────┘
                                           │
                                     Watermark
                                   (bounds state,
                                   handles late data)
```

In this notebook you will learn:
- Every built-in source: **Rate**, **RatePerMicrobatch**, **File**, **Kafka**, **Socket**
- Source options that matter in production: `maxFilesPerTrigger`, `latestFirst`, `startingOffsets`
- Every built-in sink: **memory**, **console**, **file** (Parquet / Delta), **Kafka**, **foreach**, **foreachBatch**
- Sink output modes, fault-tolerance guarantees, and when to choose each
- Watermarking internals: how the watermark value is computed, how it moves, how it releases state
- The three-way interaction between watermark, window, and output mode
- Choosing the right watermark duration for your SLA

In [ ]:
import os, time, json, shutil, random, string
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, max as _max, min as _min,
    window, current_timestamp, expr, to_timestamp, from_json,
    to_json, struct, lit, concat, when, floor, rand
)

spark = (
    SparkSession.builder
    .appName("SourcesSinksWatermarking")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE       = os.path.dirname(os.path.abspath("16-sources-sinks-watermarking.ipynb"))
DATA       = os.path.join(BASE, "data")
SCRATCH    = os.path.join(BASE, "_scratch_16")
CKPT       = os.path.join(SCRATCH, "checkpoints")
FILE_SRC   = os.path.join(SCRATCH, "file_source")
FILE_SINK  = os.path.join(SCRATCH, "file_sink")
DELTA_SINK = os.path.join(SCRATCH, "delta_sink")
LATE_SRC   = os.path.join(SCRATCH, "late_source")

for d in [CKPT, FILE_SRC, FILE_SINK, DELTA_SINK, LATE_SRC]:
    os.makedirs(d, exist_ok=True)

print(f"Spark {spark.version} ready")

### Sources Overview

A **source** is the entry point of a streaming pipeline. Spark tracks source progress (offsets) in the checkpoint so each micro-batch is processed exactly once on restart.

| Source | Format string | Fault-tolerant | Best used for |
|---|---|---|---|
| **Rate** | `"rate"` | Yes | Testing, benchmarking, demos |
| **Rate Per Microbatch** | `"rate-micro-batch"` | Yes | Deterministic testing |
| **File** | `"json"` / `"csv"` / `"parquet"` / `"orc"` / `"delta"` | Yes | File-drop pipelines, ETL |
| **Kafka** | `"kafka"` | Yes | Production event streaming |
| **Socket** | `"socket"` | **No** | Interactive demos only |

All sources are read with `spark.readStream.format(...).option(...).load()`.

### Rate Source

Generates rows at a fixed rate. Two columns: `timestamp` (event time, assigned by Spark) and `value` (0, 1, 2, …).

| Option | Default | Meaning |
|---|---|---|
| `rowsPerSecond` | 1 | Rows generated per second across all partitions |
| `rampUpTime` | 0 | Seconds to linearly ramp from 0 to `rowsPerSecond` |
| `numPartitions` | Spark default parallelism | Number of tasks generating rows |

The rate source is **fault-tolerant** — Spark tracks which values have been emitted via the checkpoint, so on restart it continues from where it left off rather than re-generating values.

In [ ]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .option("numPartitions",  2)
    .load()
)

print("Schema:"); rate_df.printSchema()

# Derive a fake transaction category from the value for more interesting demos
categories = ["FOOD", "TRAVEL", "SHOPPING", "UTILITIES", "ENTERTAINMENT"]
rate_enriched = (
    rate_df
    .withColumn("category",
        when(col("value") % 5 == 0, "FOOD")
        .when(col("value") % 5 == 1, "TRAVEL")
        .when(col("value") % 5 == 2, "SHOPPING")
        .when(col("value") % 5 == 3, "UTILITIES")
        .otherwise("ENTERTAINMENT")
    )
    .withColumn("amount", (col("value") % 490 + 10).cast("double"))
)

rate_q = (
    rate_enriched
    .writeStream.format("memory").queryName("rate_demo")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "rate"))
    .start()
)
time.sleep(4)
spark.sql("SELECT * FROM rate_demo ORDER BY value LIMIT 12").show()
rate_q.stop()

### Rate-Per-Microbatch Source

`rate-micro-batch` is similar to `rate` but generates a **fixed number of rows per micro-batch** rather than per second. The timestamp is still auto-assigned but the row count is deterministic regardless of how fast micro-batches execute. Useful for reproducible tests.

In [ ]:
rmb_df = (
    spark.readStream
    .format("rate-micro-batch")
    .option("rowsPerBatch", 50)   # exactly 50 rows every trigger
    .option("numPartitions", 2)
    .load()
)

rmb_q = (
    rmb_df
    .writeStream.format("memory").queryName("rmb_demo")
    .outputMode("append")
    .trigger(processingTime="1 second")
    .option("checkpointLocation", os.path.join(CKPT, "rmb"))
    .start()
)

time.sleep(5)
total = spark.sql("SELECT count(*) AS n FROM rmb_demo").collect()[0][0]
print(f"Rows after ~4 batches (expect ~200): {total}")
rmb_q.stop()

### File Source — Deep Dive

The file source watches a directory and processes each **new file** as a micro-batch. Spark tracks which files have been processed in the checkpoint — on restart, already-processed files are skipped.

**Key options:**

| Option | Default | Meaning |
|---|---|---|
| `maxFilesPerTrigger` | unlimited | Max new files to process per micro-batch |
| `latestFirst` | `false` | Process newest files before older ones (useful for catching up) |
| `maxFileAge` | `7 days` | Files older than this are ignored |
| `cleanSource` | `off` | `delete` or `archive` processed files |
| `sourceArchiveDir` | — | Target dir when `cleanSource=archive` |

**Schema must be provided explicitly** — Spark cannot infer schema for streaming file sources.

**Supported formats:** JSON, CSV, Parquet, ORC, text, Delta Lake.

In [ ]:
txn_schema = StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])

CATS    = ["FOOD", "TRAVEL", "SHOPPING", "UTILITIES", "ENTERTAINMENT"]
STATUES = ["SUCCESS", "SUCCESS", "SUCCESS", "DECLINED", "PENDING"]

def write_batch(idx: int, n: int = 30, base_dt: datetime = None):
    """Write n synthetic transaction rows as a JSON file."""
    if base_dt is None:
        base_dt = datetime(2024, 1, 1, 9, 0, 0) + timedelta(minutes=idx * 5)
    rows = []
    for i in range(n):
        rows.append({
            "txn_id":      f"T{idx:03d}{i:04d}",
            "customer_id": f"CUST{random.randint(1,20):04d}",
            "amount":      round(random.uniform(10, 600), 2),
            "category":    random.choice(CATS),
            "txn_ts":      (base_dt + timedelta(seconds=i)).strftime("%Y-%m-%dT%H:%M:%S"),
            "status":      random.choice(STATUES),
            "is_fraud":    random.random() < 0.05,
        })
    path = os.path.join(FILE_SRC, f"batch_{idx:03d}.json")
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"  Written {path}  ({n} rows)")

# Pre-populate the source directory
for i in range(3):
    write_batch(i)

# File source with 1 file per trigger, newest-first
file_src = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 1)
    .option("latestFirst",        "false")
    .json(FILE_SRC)
)

print(f"\nisStreaming: {file_src.isStreaming}")
file_src.printSchema()

In [ ]:
file_q = (
    file_src
    .filter(col("status") == "SUCCESS")
    .writeStream
    .format("memory").queryName("file_ingest")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "file_ingest"))
    .start()
)

time.sleep(8)   # 3 files × ~2 s each
print(f"Rows ingested: {spark.sql('SELECT count(*) FROM file_ingest').collect()[0][0]}")

# Drop a new file mid-stream — picked up on the next trigger
write_batch(3)
time.sleep(4)
print(f"Rows after new file: {spark.sql('SELECT count(*) FROM file_ingest').collect()[0][0]}")

file_q.stop()

### Kafka Source — Configuration Reference

Kafka is the production-grade streaming source. The connector reads each message as one row. The `value` column contains the raw message payload as bytes — you must deserialize it.

**Row schema produced by the Kafka source:**

| Column | Type | Description |
|---|---|---|
| `key` | `BinaryType` | Message key (bytes) |
| `value` | `BinaryType` | Message body (bytes) — deserialize with `from_json` or `.cast("string")` |
| `topic` | `StringType` | Topic the message came from |
| `partition` | `IntegerType` | Kafka partition number |
| `offset` | `LongType` | Offset within the partition |
| `timestamp` | `TimestampType` | Kafka-assigned message timestamp |
| `timestampType` | `IntegerType` | 0 = CreateTime (producer), 1 = LogAppendTime (broker) |
| `headers` | `ArrayType(StructType)` | Kafka headers (key-value metadata) |

**Essential Kafka source options:**

| Option | Example | Meaning |
|---|---|---|
| `kafka.bootstrap.servers` | `broker1:9092,broker2:9092` | Broker addresses |
| `subscribe` | `card_transactions` | Exact topic name(s), comma-separated |
| `subscribePattern` | `txn_.*` | Regex matching multiple topics |
| `assign` | `{"topic":{"0":0}}` | Explicit partition + offset assignment |
| `startingOffsets` | `"latest"` / `"earliest"` / JSON | Where to start reading |
| `endingOffsets` | `"latest"` / JSON | Only for batch reads |
| `maxOffsetsPerTrigger` | `10000` | Rate-limit: max messages per micro-batch |
| `minPartitions` | `4` | Minimum partitions Spark creates from Kafka partitions |
| `kafka.security.protocol` | `SASL_SSL` | TLS + auth for production clusters |
| `kafka.sasl.mechanism` | `PLAIN` | SASL mechanism |
| `failOnDataLoss` | `true` | Whether to fail if offsets are unavailable (e.g., topic deleted) |

In [ ]:
# ── Kafka source pattern — requires a running Kafka broker to execute ──────────
# Shown here as a reference; the rest of the notebook uses file / rate sources.

# 1. Read raw Kafka stream
# kafka_raw = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "localhost:9092")
#     .option("subscribe",               "card_transactions")
#     .option("startingOffsets",         "latest")
#     .option("maxOffsetsPerTrigger",    10_000)   # rate-limit per micro-batch
#     .load()
# )

# 2. Deserialize the value column (JSON → struct)
# parsed = (
#     kafka_raw
#     .select(
#         col("topic"),
#         col("partition"),
#         col("offset"),
#         col("timestamp").alias("kafka_ts"),
#         from_json(col("value").cast("string"), txn_schema).alias("data"),
#     )
#     .select("topic", "partition", "offset", "kafka_ts", "data.*")
# )

# 3. Use event time from the payload, not the Kafka timestamp
#    (Kafka timestamp can be delayed if the producer buffers)
# result = parsed.withWatermark("txn_ts", "10 minutes") ...

print("Kafka source pattern shown above (requires a running Kafka broker).")
print("Key steps: readStream → deserialize value with from_json → use payload event-time.")

### Sinks Overview

A **sink** is where Spark writes the results of each micro-batch. Sinks differ in:
- Which **output modes** they support
- Whether they provide **fault-tolerance / exactly-once** guarantees
- Whether they are for **development** or **production**

| Sink | Format string | Output modes | Fault-tolerant | Use for |
|---|---|---|---|---|
| **Memory** | `"memory"` | append, complete | No | Dev / testing |
| **Console** | `"console"` | append, update, complete | No | Dev / debugging |
| **File** (Parquet/JSON/CSV) | `"parquet"` etc. | append only | Yes | Immutable file pipelines |
| **Delta Lake** | `"delta"` | append, complete, update | Yes | Production, mutable tables |
| **Kafka** | `"kafka"` | append, update, complete | Yes | Event forwarding |
| **foreach** | `.foreach(ForeachWriter)` | append, update, complete | Depends on writer | Row-level custom logic |
| **foreachBatch** | `.foreachBatch(func)` | append, update, complete | Depends on func | Batch-level custom logic |

### Console Sink

The console sink prints each micro-batch to stdout. Supports all three output modes, making it ideal for quickly inspecting what each mode produces.

| Option | Default | Meaning |
|---|---|---|
| `numRows` | 20 | Max rows printed per batch |
| `truncate` | true | Whether to truncate long strings |

The console sink is **not fault-tolerant** — on restart it simply prints whatever the next micro-batch produces.

In [ ]:
# Pre-populate more data for sink demos
for i in range(4, 7):
    write_batch(i)

src = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 2)
    .json(FILE_SRC)
)

console_q = (
    src
    .filter(col("is_fraud") == True)
    .select("txn_id", "customer_id", "amount", "category", "txn_ts")
    .writeStream
    .format("console")
    .option("numRows",  5)
    .option("truncate", False)
    .outputMode("append")
    .trigger(once=True)
    .option("checkpointLocation", os.path.join(CKPT, "console"))
    .start()
)
console_q.awaitTermination()
print("Console sink complete.")

### File Sink — Parquet

The file sink writes each micro-batch as one or more new files in a directory. It supports **append mode only** — existing files are never modified. Each micro-batch produces a new set of files. This makes the file sink naturally immutable and idempotent on restart (the checkpoint tracks which batches have been committed; already-committed batches are not re-written).

**Partitioning**: use `partitionBy(col, ...)` to write Hive-style partitioned directories (`year=2024/month=01/`). Downstream readers can then use partition pruning.

**Compaction problem**: many small files accumulate over time (one file set per micro-batch). Use Delta Lake as the sink to avoid this — Delta compacts automatically.

In [ ]:
src2 = (
    spark.readStream
    .schema(txn_schema)
    .json(FILE_SRC)
)

parquet_path = os.path.join(FILE_SINK, "transactions")

parquet_q = (
    src2
    .filter(col("status") == "SUCCESS")
    .withColumn("year",     col("txn_ts").cast("date").cast("string").substr(1, 4))
    .withColumn("month",    col("txn_ts").cast("date").cast("string").substr(6, 2))
    .withColumn("category", col("category"))
    .writeStream
    .format("parquet")
    .option("path",               parquet_path)
    .option("checkpointLocation", os.path.join(CKPT, "parquet_sink"))
    .partitionBy("year", "month", "category")
    .outputMode("append")
    .trigger(once=True)
    .start()
)
parquet_q.awaitTermination()

# Inspect the written partitions
written = spark.read.parquet(parquet_path)
print(f"Rows written: {written.count()}")
written.groupBy("year", "month", "category").count().orderBy("year","month","category").show()

### Delta Lake Sink — Mutable Streaming Output

Delta Lake is the recommended production sink because:
- Supports **append, complete, and update** output modes
- Provides **exactly-once** via transactional commits (no duplicate files on restart)
- Avoids the small-file problem through background `OPTIMIZE` / auto-compaction
- Allows downstream readers to query the table while it is being written (snapshot isolation)
- **Update mode** writes only changed rows each trigger — efficient for aggregation results

For update/complete mode with Delta, Spark uses the **merge** capability internally to upsert changed rows without rewriting the full table.

In [ ]:
src3 = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 2)
    .json(FILE_SRC)
)

delta_path = os.path.join(DELTA_SINK, "txn_summary")

# Aggregation → Delta sink in complete mode (full result table written each trigger)
summary_agg = (
    src3
    .filter(col("status") == "SUCCESS")
    .groupBy("category")
    .agg(
        count("txn_id").alias("txn_count"),
        _sum("amount").alias("total_spend"),
        avg("amount").alias("avg_spend"),
    )
)

delta_q = (
    summary_agg
    .writeStream
    .format("delta")
    .option("path",               delta_path)
    .option("checkpointLocation", os.path.join(CKPT, "delta_sink"))
    .outputMode("complete")
    .trigger(once=True)
    .start()
)
delta_q.awaitTermination()

result = spark.read.format("delta").load(delta_path)
print("Delta sink contents:")
result.orderBy(col("total_spend").desc()).show()

### Kafka Sink — Reference

The Kafka sink publishes each output row as a Kafka message. The DataFrame must have a `value` column (BinaryType or StringType). An optional `key` column sets the message key; optional `topic` column overrides the target topic per row.

```python
from pyspark.sql.functions import to_json, struct, col

fraud_stream = (
    parsed_stream
    .filter(col("is_fraud") == True)
    .select(
        col("txn_id").alias("key"),              # Kafka message key
        to_json(struct("*")).alias("value"),     # serialize entire row as JSON
    )
)

fraud_stream.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic",                   "fraud_alerts")  # single target topic
    .option("checkpointLocation",      "/ckpt/fraud")
    .outputMode("append")
    .start()
```

**Dynamic topic routing** — include a `topic` column in the DataFrame to route different rows to different topics:

```python
.withColumn("topic",
    when(col("amount") > 1000, "high_value_txns")
    .otherwise("normal_txns")
)
# Do NOT set the "topic" option in writeStream — use the column instead
```

### foreachBatch Sink — Multi-Table Writes

`foreachBatch(func)` gives you a regular (static) DataFrame for each micro-batch. The primary use cases:

1. **Multi-table writes** — write one batch to two different sinks (e.g., Delta for analytics + Kafka for alerting)
2. **Conditional fan-out** — route rows to different tables based on content
3. **Custom deduplication** — use `batch_id` + a MERGE statement for exactly-once
4. **Non-Spark sinks** — call a REST API, write to Redis, send emails

**Idempotency pattern**: check `batch_id` before writing — if the job restarts, the same `batch_id` will be replayed. Use a MERGE or an idempotent write to prevent duplicates.

In [ ]:
SUCCESS_PATH = os.path.join(SCRATCH, "success_txns")
FRAUD_PATH   = os.path.join(SCRATCH, "fraud_txns")

batch_log = []   # track what each batch wrote

def write_to_two_sinks(batch_df, batch_id):
    """
    foreachBatch handler:
      - write successful transactions to a Parquet archive
      - write fraud transactions to a separate Delta table
    """
    batch_df.persist()  # cache — used twice below

    success = batch_df.filter((col("status") == "SUCCESS") & (col("is_fraud") == False))
    fraud   = batch_df.filter(col("is_fraud") == True)

    s_count = success.count()
    f_count = fraud.count()

    # Sink 1 — Parquet archive for successful non-fraud transactions
    if s_count > 0:
        success.write.mode("append").parquet(SUCCESS_PATH)

    # Sink 2 — Delta table for fraud alerts
    if f_count > 0:
        fraud.write.format("delta").mode("append").save(FRAUD_PATH)

    batch_log.append({"batch_id": batch_id, "success": s_count, "fraud": f_count})
    print(f"  batch {batch_id}: {s_count} success rows → Parquet | {f_count} fraud rows → Delta")

    batch_df.unpersist()


src4 = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 2)
    .json(FILE_SRC)
)

foreach_q = (
    src4
    .writeStream
    .foreachBatch(write_to_two_sinks)
    .option("checkpointLocation", os.path.join(CKPT, "foreach"))
    .trigger(once=True)
    .start()
)
foreach_q.awaitTermination()

print("\nBatch log:")
for entry in batch_log:
    print(f"  {entry}")

print(f"\nSuccess Parquet rows : {spark.read.parquet(SUCCESS_PATH).count()}")
print(f"Fraud Delta rows     : {spark.read.format('delta').load(FRAUD_PATH).count()}")

### Watermarking — How It Works Internally

Watermarking solves two problems simultaneously:

1. **Late data** — events arrive out of order; how long should Spark wait before finalizing a window?
2. **Bounded state** — windowed aggregations accumulate state indefinitely without a cleanup rule

**The watermark formula:**

```
watermark = max(event_time seen so far) - watermark_delay
```

Any event whose event time is **before the watermark** is considered late and dropped. Any window whose end time is **before the watermark** is closed — its state can be freed and its result emitted.

**Step-by-step example** — watermark delay = 10 minutes, window = 5 minutes:

```text
Micro-batch 1: events at 09:01, 09:03, 09:07
  max event time = 09:07   →   watermark = 09:07 - 10 min = 08:57
  Windows [09:00,09:05) and [09:05,09:10) are OPEN (end > 08:57)

Micro-batch 2: events at 09:12, 09:18
  max event time = 09:18   →   watermark = 09:08
  Window [09:00,09:05) ends at 09:05 < watermark 09:08 → CLOSED, result emitted

Micro-batch 3: late event arrives at 09:02 (from 9 minutes ago)
  current watermark = 09:08   →   09:02 < 09:08 → DROPPED

Micro-batch 4: late event at 09:06 (2 minutes late, still within 10-min watermark)
  current watermark = 09:08   →   09:06 < 09:08 → DROPPED (just over the threshold)
```

**Important**: the watermark is updated based on the **maximum event time seen across all partitions** in a micro-batch, not the maximum per partition. Spark takes the minimum watermark across all tasks to guarantee correctness.

In [ ]:
# Demonstrate watermark tracking via lastProgress

wm_src = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 1)
    .json(FILE_SRC)
)

wm_agg = (
    wm_src
    .withWatermark("txn_ts", "10 minutes")
    .groupBy(window(col("txn_ts"), "5 minutes"), col("category"))
    .agg(count("txn_id").alias("txn_count"))
    .select(
        col("window.start").alias("win_start"),
        col("window.end").alias("win_end"),
        "category", "txn_count"
    )
)

wm_q = (
    wm_agg
    .writeStream.format("memory").queryName("wm_demo")
    .outputMode("update")
    .option("checkpointLocation", os.path.join(CKPT, "wm_demo"))
    .start()
)

# Poll a few batches and print the watermark after each
for tick in range(4):
    time.sleep(4)
    p = wm_q.lastProgress
    if p:
        wm_val = p.get("eventTime", {}).get("watermark", "(not yet set)")
        max_et = p.get("eventTime", {}).get("max",       "(not yet set)")
        print(f"Batch {p['batchId']:>3} | max event time = {max_et} | watermark = {wm_val}")

wm_q.stop()

print("\nWindowed results accumulated:")
spark.sql("SELECT * FROM wm_demo ORDER BY win_start, category").show(20, truncate=False)

### Late-Data Behaviour — Accepted vs Dropped

This section generates data with deliberate late arrivals to show exactly what Spark accepts and drops.

In [ ]:
def write_late_batch(filename: str, rows: list):
    """Write a batch with explicit timestamps (including deliberately late ones)."""
    path = os.path.join(LATE_SRC, filename)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"  Written {filename}: {[r['txn_ts'] for r in rows]}")

def make_row(txn_id, ts_str, cat="FOOD"):
    return {"txn_id": txn_id, "customer_id": "CUST0001",
            "amount": 100.0, "category": cat,
            "txn_ts": ts_str, "status": "SUCCESS", "is_fraud": False}

# Batch A: events in the 09:00 window (on-time)
write_late_batch("a_ontime.json", [
    make_row("A1", "2024-01-01T09:01:00"),
    make_row("A2", "2024-01-01T09:02:00"),
    make_row("A3", "2024-01-01T09:04:00"),
])

# Batch B: events push max event time to 09:22 → watermark becomes 09:12
# Window [09:00-09:05) ends at 09:05 < watermark 09:12 → it will be CLOSED
write_late_batch("b_advance.json", [
    make_row("B1", "2024-01-01T09:20:00"),
    make_row("B2", "2024-01-01T09:22:00"),
])

# Batch C: late event at 09:03 — this is BEFORE watermark 09:12 → DROPPED
write_late_batch("c_late.json", [
    make_row("C1", "2024-01-01T09:03:00", "TRAVEL"),   # 19 min late — DROPPED
    make_row("C2", "2024-01-01T09:15:00", "SHOPPING"), # within watermark — ACCEPTED
])

In [ ]:
late_schema = txn_schema

late_src = (
    spark.readStream
    .schema(late_schema)
    .option("maxFilesPerTrigger", 1)   # process one file per micro-batch
    .json(LATE_SRC)
)

late_agg = (
    late_src
    .withWatermark("txn_ts", "10 minutes")          # tolerate up to 10 min of lateness
    .groupBy(window(col("txn_ts"), "5 minutes"), col("category"))
    .agg(count("txn_id").alias("n"), _sum("amount").alias("total"))
    .select(
        col("window.start").alias("win_start"),
        col("window.end").alias("win_end"),
        "category", "n", "total"
    )
)

late_q = (
    late_agg
    .writeStream.format("memory").queryName("late_demo")
    .outputMode("update")   # emit rows as they change
    .option("checkpointLocation", os.path.join(CKPT, "late_demo"))
    .start()
)

# Let all 3 files be processed (one per batch)
time.sleep(18)

p = late_q.lastProgress
if p:
    et = p.get("eventTime", {})
    print(f"Final watermark  : {et.get('watermark', 'n/a')}")
    print(f"Max event time   : {et.get('max', 'n/a')}")

print("\nAccumulated window results:")
spark.sql("SELECT * FROM late_demo ORDER BY win_start, category").show(truncate=False)
print("Expected: C1 (TRAVEL at 09:03) is DROPPED — its window closed before that event arrived.")
print("Expected: C2 (SHOPPING at 09:15) is ACCEPTED — within watermark.")

late_q.stop()

### Watermark + Output Mode Compatibility

The output mode and watermark interact to determine when results are finalized and state is freed:

| Output mode | Watermark required? | When is a window result emitted? | When is state freed? |
|---|---|---|---|
| **complete** | No | Every trigger — full table rewritten | Never (all state kept forever) |
| **update** | Recommended | Every trigger for rows that changed | After watermark passes window end |
| **append** | **Yes** | Only after watermark passes window end (finalized) | After emission |

```text
append mode with watermark:
  Window [09:00–09:05) is NOT emitted until the watermark passes 09:05.
  Once emitted, the result is final — no updates ever.
  → Best for sinks that do not support updates (Parquet files, Kafka)

update mode with watermark:
  Window result is emitted every trigger if any rows changed.
  State is freed once watermark passes window end.
  → Best for sinks that support upserts (Delta Lake)

complete mode (no watermark):
  Full result table rewritten every trigger.
  State grows forever — only suitable for small, bounded result sets.
```

In [ ]:
# Side-by-side: same aggregation in update vs complete mode

def make_windowed_agg(stream, output_mode, query_name, ckpt_name):
    return (
        stream
        .withWatermark("txn_ts", "5 minutes")
        .groupBy(window(col("txn_ts"), "5 minutes"))
        .agg(count("txn_id").alias("n"), _sum("amount").alias("total"))
        .select(
            col("window.start").alias("win_start"),
            col("window.end").alias("win_end"),
            "n", "total"
        )
        .writeStream
        .format("memory").queryName(query_name)
        .outputMode(output_mode)
        .option("checkpointLocation", os.path.join(CKPT, ckpt_name))
        .start()
    )

src_u = spark.readStream.schema(txn_schema).option("maxFilesPerTrigger", 2).json(FILE_SRC)
src_c = spark.readStream.schema(txn_schema).option("maxFilesPerTrigger", 2).json(FILE_SRC)

q_update   = make_windowed_agg(src_u, "update",   "mode_update",   "mode_update")
q_complete = make_windowed_agg(src_c, "complete",  "mode_complete", "mode_complete")

time.sleep(14)

print("=== UPDATE mode (only changed/new windows per trigger) ===")
spark.sql("SELECT * FROM mode_update ORDER BY win_start").show(truncate=False)

print("=== COMPLETE mode (full table every trigger) ===")
spark.sql("SELECT * FROM mode_complete ORDER BY win_start").show(truncate=False)

q_update.stop()
q_complete.stop()

### Choosing the Watermark Duration

The watermark delay is a **business SLA decision**, not a technical one. It answers: "How late can data arrive and still be counted?"

```text
Longer watermark (e.g., 2 hours):
  ✓ Accepts more late data — higher accuracy
  ✗ State held longer — more executor memory consumed
  ✗ Results delayed by the watermark duration (append mode)

Shorter watermark (e.g., 1 minute):
  ✓ State freed quickly — less memory
  ✓ Results finalized sooner (append mode)
  ✗ Late events dropped — lower accuracy for stragglers
```

**Practical guidelines:**

| Scenario | Suggested watermark |
|---|---|
| Kafka (healthy, low-latency) | 1–5 minutes |
| Kafka (cross-DC replication lag) | 10–30 minutes |
| Mobile app events (offline → reconnect) | 1–24 hours |
| IoT sensors (intermittent connectivity) | 1–6 hours |
| File drops from upstream batch jobs | Duration of batch interval + buffer |

**How to measure**: run the pipeline in `update` mode (no watermark) for a day, then look at `event_time.max - event_time.min` across batches at the p99 level. That tail latency is your minimum viable watermark.

In [ ]:
# Measure event-time spread in each micro-batch to inform watermark choice
spread_q = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 1)
    .json(FILE_SRC)
    .writeStream
    .format("memory").queryName("spread_check")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "spread"))
    .start()
)

time.sleep(16)
spread_q.stop()

# After collecting, compute the spread per calendar minute (simulate per-batch)
from pyspark.sql.functions import date_trunc, datediff, unix_timestamp

spread = (
    spark.sql("SELECT txn_ts FROM spread_check")
    .agg(
        _min("txn_ts").alias("earliest"),
        _max("txn_ts").alias("latest"),
        count("*").alias("total_rows"),
    )
    .withColumn(
        "span_minutes",
        (unix_timestamp(col("latest")) - unix_timestamp(col("earliest"))) / 60
    )
)
spread.show(truncate=False)
print("Watermark recommendation: set delay to at least the span_minutes value above, with a safety buffer.")

### Stream-Static Join

A **stream-static join** enriches each incoming row with data from a static (batch) DataFrame — typically a dimension table. The static side is broadcast to all executors. This is a stateless operation and supports all output modes.

**Rules:**
- The static DataFrame is re-read every micro-batch — changes to the static data are eventually picked up
- No watermark needed (the static side never generates late data)
- The join key must be present in both sides
- Left outer join is safe; right outer and full outer are not supported with streaming on the left

In [ ]:
# Load static customers dimension table
customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",  StringType(),    False),
    StructField("full_name",    StringType(),    False),
    StructField("city",         StringType(),    True),
    StructField("state",        StringType(),    True),
    StructField("kyc_verified", BooleanType(),   False),
    StructField("created_at",   TimestampType(), False),
])).csv(f"{DATA}/customers")

stream_src = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 2)
    .json(FILE_SRC)
)

enriched = (
    stream_src
    .join(customers.select("customer_id", "full_name", "city", "kyc_verified"),
          on="customer_id", how="left")
    .select(
        "txn_id", "customer_id", col("full_name"),
        col("city"), "amount", "category", "txn_ts", "status"
    )
)

enrich_q = (
    enriched
    .writeStream.format("memory").queryName("enriched_txns")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "enrich"))
    .trigger(once=True)
    .start()
)
enrich_q.awaitTermination()

print("Enriched stream results:")
spark.sql("""
    SELECT txn_id, customer_id, full_name, city, amount, category
    FROM   enriched_txns
    WHERE  full_name IS NOT NULL
    LIMIT  10
""").show(truncate=False)

### Source & Sink Fault-Tolerance Matrix

End-to-end exactly-once requires **both** the source and the sink to support it:

| Source | Sink | End-to-end guarantee |
|---|---|---|
| Kafka (with offsets) | Delta Lake | Exactly-once |
| Kafka (with offsets) | Kafka (idempotent producer) | Exactly-once |
| File source (tracked by checkpoint) | Parquet / Delta | Exactly-once |
| Rate source | Memory / Console | At-least-once (sinks not durable) |
| Socket | Any | At-most-once (source not replayable) |
| Any | `foreach` / `foreachBatch` | Depends on your implementation |

**How exactly-once works in practice:**
1. Source: Spark records offsets before processing each micro-batch
2. Processing: the micro-batch computes results
3. Sink commit: the sink writes results transactionally
4. Checkpoint: offsets marked committed

If the job crashes between steps 2 and 4, the same micro-batch is reprocessed on restart. The sink's idempotence (Delta transactional commit, Kafka idempotent producer) ensures no duplicates are written.

### Summary

**Sources** read data into a streaming DataFrame:
- **Rate** / **RatePerMicrobatch** — synthetic generators for testing; no external dependencies
- **File** — watches a directory; processes each new file as a micro-batch; requires explicit schema; `maxFilesPerTrigger` controls throughput
- **Kafka** — production event streaming; `value` column is raw bytes; deserialize with `from_json`; use `maxOffsetsPerTrigger` to rate-limit; prefer payload event time over Kafka timestamp
- **Socket** — dev-only; not fault-tolerant

**Sinks** receive the results of each micro-batch:
- **memory** / **console** — dev/testing only; not fault-tolerant
- **File (Parquet/ORC/JSON)** — append-only; accumulates small files; no update support
- **Delta Lake** — best production sink; supports all output modes; exactly-once; no small-file problem
- **Kafka** — forwards results downstream; needs `key` + `value` columns
- **foreachBatch** — custom logic per micro-batch; multi-table writes; idempotency via `batch_id`

**Watermarking** (`withWatermark(event_time_col, delay)`):
- Watermark = max(event time seen) − delay
- Events older than the watermark are **dropped**
- Windows ending before the watermark are **closed** and their state **freed**
- Required for `append` output mode with windowed aggregations
- Recommended for `update` mode to bound state growth
- Longer delay = more late data accepted, more memory consumed, later result finalization
- Choose delay based on your p99 event lateness, measured from real pipeline data

**Output mode × watermark rules:**
- `complete` — no watermark needed; full table every trigger; state never freed
- `update` + watermark — only changed rows per trigger; state freed after watermark passes window end
- `append` + watermark — only finalized (closed) windows emitted; best for immutable sinks

In [ ]:
# Cleanup all scratch directories
if os.path.exists(SCRATCH):
    shutil.rmtree(SCRATCH)
    print(f"Removed scratch directory: {SCRATCH}")
print("Cleanup complete.")